# Day 93 — Streamlit Interactive Client Dashboard
**Month 5 | Week 3 | Python + Streamlit**

> **Real-world framing:** You've built the pipeline (Days 90–92). Now the client says:
> *"Can I interact with this data myself — filter by region, pick a date range, download a report?"*
> That's what Streamlit solves. One Python file → a live web dashboard that non-technical
> clients can use. This is a direct Upwork differentiator: most analysts deliver static PDFs.
> You'll deliver a shareable link.

**Deliverable:** `Day93_dashboard.py` — a runnable Streamlit app
**Run with:** `streamlit run Day93_dashboard.py`
**Score:** 80 pts + 10 bonus


## Section 0 — Setup & Dataset Generation
*Run once. This creates `techmart_day93.csv` that your dashboard will load.*

In [ ]:
# Section 0: Generate dataset + install Streamlit
# DO NOT MODIFY this cell — raw data source

import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'streamlit', 'pandas',
                'matplotlib', 'seaborn', '--quiet'], check=True)

import pandas as pd
import numpy as np
import os

np.random.seed(93)

n = 150
regions     = ['North', 'South', 'East', 'West']
categories  = ['Electronics', 'Accessories', 'Clothing', 'Home & Kitchen', 'Sports']
products    = {
    'Electronics':     ['Smartphone', 'Laptop', 'Tablet', 'Smart Watch'],
    'Accessories':     ['Earphones', 'USB Cable', 'Phone Cover', 'Screen Guard'],
    'Clothing':        ['T-Shirt', 'Jeans', 'Kurta', 'Jacket'],
    'Home & Kitchen':  ['Mixer', 'Pressure Cooker', 'Water Bottle', 'Air Fryer'],
    'Sports':          ['Cricket Bat', 'Yoga Mat', 'Running Shoes', 'Skipping Rope'],
}
segments    = ['New', 'Returning', 'Premium', 'Wholesale']
unit_prices = {
    'Smartphone': 18999, 'Laptop': 54999, 'Tablet': 24999, 'Smart Watch': 8999,
    'Earphones': 1299,   'USB Cable': 499,  'Phone Cover': 249, 'Screen Guard': 199,
    'T-Shirt': 599,      'Jeans': 1499,     'Kurta': 899,       'Jacket': 2499,
    'Mixer': 3299,       'Pressure Cooker': 2199, 'Water Bottle': 399, 'Air Fryer': 5999,
    'Cricket Bat': 1999, 'Yoga Mat': 899,   'Running Shoes': 3499, 'Skipping Rope': 299,
}

cats  = np.random.choice(categories, n)
prods = [np.random.choice(products[c]) for c in cats]
qtys  = np.random.randint(1, 6, n)
prices= np.array([unit_prices[p] for p in prods])
revs  = qtys * prices

dates = pd.date_range('2024-01-01', '2024-12-31', periods=n).normalize()
dates = dates + pd.to_timedelta(np.random.randint(0, 1, n), unit='D')

df = pd.DataFrame({
    'order_id':        [f'ORD-{9300+i}' for i in range(n)],
    'date':            dates,
    'region':          np.random.choice(regions, n),
    'category':        cats,
    'product':         prods,
    'quantity':        qtys,
    'unit_price':      prices,
    'revenue':         revs,
    'customer_segment': np.random.choice(segments, n),
})

df['date'] = pd.to_datetime(df['date'])
df.to_csv('techmart_day93.csv', index=False)
print(f"Dataset saved: {df.shape[0]} rows × {df.shape[1]} cols")
print(df.head(3).to_string())
print(f"\nDate range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Revenue range: ₹{df['revenue'].min():,} to ₹{df['revenue'].max():,}")
print(f"Total revenue: ₹{df['revenue'].sum():,.0f}")


---
## Section 1 — Practice Guide

**Your task:** Build `Day93_dashboard.py` by completing each task block below.
Write your code in each cell → verify it works in isolation → assemble the final file in **Section 5**.

### Task Map

| Task | Topic | Points |
|------|-------|--------|
| T1 | App config + layout + sidebar header | 8 |
| T2 | Load data + sidebar filters (Region, Category, Date) | 12 |
| T3 | KPI metric cards (Revenue, Orders, Avg Order Value) | 12 |
| T4 | Bar chart — Revenue by Category | 12 |
| T5 | Line chart — Monthly Revenue Trend | 12 |
| T6 | Interactive filtered data table | 8 |
| T7 | CSV download button | 8 |
| W1 | Written: Interview answer + README blurb | 8 |
| **B★** | Bonus: Custom CSS header + Pie chart | +10 |
| **Total** | | **80 + 10** |

---
### Professional Rules for This Day

1. **Print → Read → Write**: Before writing any number in a text string (e.g., `st.metric`), compute it and `print()` it first. Copy the exact value. No mental math.
2. **`dropna` on anchor column** before any analysis — always the first operation after `pd.read_csv`.
3. **Filter before compute** — all metrics and charts must use `df_filtered`, never `df`.
4. **NRA format** in any text insight: Number + Reason + Action.


### T1 — App Config + Layout + Sidebar Header (8 pts)

In [ ]:
# T1: App setup skeleton
# Write your plain-English comments first, then code each line

# Concept: st.set_page_config must be the FIRST Streamlit call in the file
# It controls the browser tab title, icon, and layout width

# YOUR CODE HERE — write the following, in order:
# 1. Import streamlit as st, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
# 2. st.set_page_config: title='TechMart India | Analytics Dashboard',
#    layout='wide', page_icon='📊'
# 3. st.title with a 📊 emoji prefix
# 4. st.subheader — a one-line business framing sentence (not a label, a sentence)
# 5. st.divider()
# 6. Sidebar: st.sidebar.header('🎛️ Filters')

# Verify: paste into app.py, run `streamlit run Day93_dashboard.py`
# You should see the title + empty sidebar filter panel


### T2 — Load Data + Sidebar Filters (12 pts)

In [ ]:
# T2: Load data and build sidebar filters
# This is the most important task — bad filtering = all downstream metrics wrong

# Step 1: Load CSV
# df = pd.read_csv('techmart_day93.csv', parse_dates=['date'])

# Step 2: Drop rows where 'order_id' is null — anchor column rule
# df = df.dropna(subset=['order_id']).reset_index(drop=True)

# Step 3: Build sidebar filters
# Filter 1 — Region multiselect
#   regions_available = sorted(df['region'].unique())
#   selected_regions = st.sidebar.multiselect('Region', regions_available, default=regions_available)

# Filter 2 — Category multiselect
#   categories_available = sorted(df['category'].unique())
#   selected_cats = st.sidebar.multiselect('Category', categories_available, default=categories_available)

# Filter 3 — Date range slider
#   min_date = df['date'].min().date()
#   max_date = df['date'].max().date()
#   date_range = st.sidebar.date_input('Date Range', value=[min_date, max_date],
#                min_value=min_date, max_value=max_date)

# Step 4: Apply filters
# df_filtered = df[
#     (df['region'].isin(selected_regions)) &
#     (df['category'].isin(selected_cats)) &
#     (df['date'].dt.date >= date_range[0]) &
#     (df['date'].dt.date <= date_range[1])
# ].copy()

# Step 5: Show row count in sidebar
# st.sidebar.markdown(f'**{len(df_filtered)} orders** match filters')

# YOUR CODE: write all steps above as executable code below this comment


### T3 — KPI Metric Cards (12 pts)

In [ ]:
# T3: Three KPI cards using st.metric
# st.metric shows a big number + an optional delta (vs baseline)

# Required metrics (all from df_filtered):
# KPI 1 — Total Revenue (₹, formatted with commas)
# KPI 2 — Total Orders (row count of df_filtered)
# KPI 3 — Avg Order Value = Total Revenue / Total Orders (₹, 2 decimal places)

# Layout: use st.columns(3) to place all three side by side

# CRITICAL: Compute and print each value BEFORE writing the st.metric string
# Example:
#   total_revenue = df_filtered['revenue'].sum()
#   print(total_revenue)   ← read this number, then write it below
#   col1.metric('💰 Total Revenue', f'₹{total_revenue:,.0f}')

# For delta: compare filtered avg order value vs full dataset avg order value
# baseline_aov = df['revenue'].mean()
# filtered_aov = df_filtered['revenue'].mean() if len(df_filtered) > 0 else 0
# delta_aov = filtered_aov - baseline_aov

# YOUR CODE HERE


### T4 — Bar Chart: Revenue by Category (12 pts)

In [ ]:
# T4: Horizontal bar chart — Revenue by Category
# Use matplotlib (not st.bar_chart — that gives less control)

# Steps:
# 1. Aggregate: df_cat = df_filtered.groupby('category')['revenue'].sum().sort_values(ascending=True)
# 2. Print df_cat — read the exact numbers before writing any text
# 3. Build figure: fig, ax = plt.subplots(figsize=(8, 4))
# 4. ax.barh(df_cat.index, df_cat.values, color='#1F3864')
# 5. Format x-axis as ₹ values: ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'₹{x/1000:.0f}K'))
# 6. ax.set_title('Revenue by Category', fontsize=13, fontweight='bold', color='#1F3864')
# 7. ax.set_xlabel('Revenue (₹)')
# 8. ax.spines[['top','right']].set_visible(False)
# 9. plt.tight_layout()
# 10. st.subheader('Revenue by Category')
# 11. st.pyplot(fig)

# Below chart: one NRA insight in st.markdown
# Template: "**[Category]** leads with ₹[exact number] in revenue,
# driven by [reason], suggesting [action]."

# YOUR CODE HERE


### T5 — Line Chart: Monthly Revenue Trend (12 pts)

In [ ]:
# T5: Monthly revenue trend line chart
# This shows seasonality — clients always ask 'which months are strong?'

# Steps:
# 1. Extract month: df_filtered['month'] = df_filtered['date'].dt.to_period('M')
# 2. Aggregate: df_monthly = df_filtered.groupby('month')['revenue'].sum().reset_index()
# 3. df_monthly['month'] = df_monthly['month'].astype(str)  # convert Period to string for plotting
# 4. Print df_monthly — read the peak month EXACTLY
# 5. Build figure: fig2, ax2 = plt.subplots(figsize=(10, 4))
# 6. ax2.plot(df_monthly['month'], df_monthly['revenue'], marker='o', color='#2E5596', linewidth=2)
# 7. ax2.fill_between(df_monthly['month'], df_monthly['revenue'], alpha=0.15, color='#2E5596')
# 8. ax2.set_title('Monthly Revenue Trend', fontsize=13, fontweight='bold', color='#1F3864')
# 9. Rotate x-axis labels: plt.xticks(rotation=45, ha='right')
# 10. Format y-axis with ₹K notation (same as T4)
# 11. ax2.spines[['top','right']].set_visible(False)
# 12. plt.tight_layout()
# 13. st.subheader('Monthly Revenue Trend')
# 14. st.pyplot(fig2)

# Below chart: one NRA insight
# Template: "[Month] was the peak at ₹[exact number], driven by [reason], 
# suggesting [action] in [Month+1/similar period]."

# YOUR CODE HERE


### T6 — Interactive Filtered Data Table (8 pts)

In [ ]:
# T6: Show the filtered dataframe in an interactive table
# Clients love being able to scroll and inspect the raw orders

# Steps:
# 1. st.subheader('📋 Order-Level Detail')
# 2. Show row count: st.caption(f'Showing {len(df_filtered)} orders')
# 3. Display with st.dataframe(df_filtered, use_container_width=True, height=300)
# 4. Format column display — rename columns for client readability:
#    display_cols = df_filtered[['order_id','date','region','category','product',
#                                'quantity','unit_price','revenue','customer_segment']].copy()
#    display_cols.columns = ['Order ID','Date','Region','Category','Product',
#                            'Qty','Unit Price (₹)','Revenue (₹)','Segment']
#    st.dataframe(display_cols, use_container_width=True, height=300)

# YOUR CODE HERE


### T7 — CSV Download Button (8 pts)

In [ ]:
# T7: Download button for filtered data
# This is a key client-facing feature — they can export exactly what they filtered

# Steps:
# 1. Convert df_filtered to CSV bytes:
#    csv_bytes = df_filtered.to_csv(index=False).encode('utf-8')

# 2. st.download_button(
#        label='⬇️ Download Filtered Data (CSV)',
#        data=csv_bytes,
#        file_name='techmart_filtered_report.csv',
#        mime='text/csv',
#        help='Downloads the current filtered view as a CSV file'
#    )

# 3. Add a footer line below the button:
#    st.caption('TechMart India Analytics Dashboard | Built with Streamlit | deepanshu0110')

# YOUR CODE HERE


### W1 — Written Section (8 pts)

In [ ]:
# W1: Two written outputs — answer both in markdown cells or as print() below

# W1a (4 pts) — Interview answer:
# Question: "A client asks you to share your analysis. What do you give them?"
# Write a 3-4 sentence answer as a professional data analyst.
# Must include: what Streamlit is, why it's better than a PDF/Excel, 
# one specific feature (filter/download) and its business value.

# W1b (4 pts) — README blurb (3-4 lines):
# Write the GitHub README description for this dashboard project.
# Format: project name → what it does → tech stack → how to run it
# This should sound like a real portfolio README, not a homework submission.

# Write your answers below as print() statements or in a markdown cell

print("W1a — Interview Answer:")
print("YOUR ANSWER HERE")
print()
print("W1b — README Blurb:")
print("YOUR ANSWER HERE")


### B★ — Bonus: Custom CSS Header + Pie Chart (+10 pts)

In [ ]:
# BONUS (optional): Two additions to the dashboard

# B1 (5 pts) — Custom CSS header banner:
# Inject CSS using st.markdown with unsafe_allow_html=True
# Add a styled header div: dark blue background, white text, centered, padding
# Example:
# st.markdown('''
# <div style="background-color:#1F3864; padding:16px; border-radius:8px; text-align:center">
#     <h2 style="color:white; margin:0">TechMart India Analytics Dashboard</h2>
#     <p style="color:#D6E4F0; margin:4px 0 0 0">Powered by Python + Streamlit</p>
# </div>
# ''', unsafe_allow_html=True)

# B2 (5 pts) — Pie chart: Revenue by Customer Segment
# df_seg = df_filtered.groupby('customer_segment')['revenue'].sum()
# Print df_seg first — read each value
# Build pie chart with:
#   colors = ['#1F3864','#2E5596','#4A90D9','#A8C4E0']
#   ax.pie(df_seg.values, labels=df_seg.index, autopct='%1.1f%%', colors=colors,
#          startangle=140, wedgeprops={'edgecolor':'white','linewidth':1.5})
#   ax.set_title('Revenue by Customer Segment', fontsize=12, fontweight='bold', color='#1F3864')
# One NRA insight below the chart.

# YOUR CODE HERE


---
## Section 2 — Concept Notes

### What is Streamlit?

Streamlit is a Python library that converts a `.py` script into an interactive web app — 
no HTML, no JavaScript, no backend needed. You write Python; Streamlit handles the frontend.

**Why clients pay for it:**
- Non-technical stakeholders can filter data themselves → saves analyst time
- Shareable URL (via Streamlit Community Cloud — free) → instant demo link for Upwork proposals
- Live data updates when connected to a database or CSV that refreshes

### Key Components Used Today

| Component | What it does | When to use |
|-----------|-------------|-------------|
| `st.set_page_config` | Sets browser tab title, icon, layout | First line always |
| `st.sidebar.multiselect` | Dropdown filter with multi-select | Categorical filters |
| `st.sidebar.date_input` | Date range picker | Time-based filtering |
| `st.metric` | Big KPI number with delta indicator | Executive summary |
| `st.columns(n)` | Side-by-side layout | KPI cards, comparison |
| `st.pyplot(fig)` | Display matplotlib/seaborn chart | Custom charts |
| `st.dataframe` | Interactive scrollable table | Raw data inspection |
| `st.download_button` | File download trigger | CSV/Excel export |
| `st.markdown(..., unsafe_allow_html=True)` | Custom HTML/CSS | Branding, styled banners |

### The Render Loop

Every time a user interacts with any widget (slider, multiselect, button), **Streamlit 
re-runs the entire script from top to bottom**. This means:
- Always compute `df_filtered` after reading all sidebar inputs
- Never put sidebar inputs *after* chart code — order matters
- Expensive computations should be cached with `@st.cache_data`

### Filter → Compute → Display Pattern

```
Load raw data
    ↓
Apply sidebar filters → df_filtered
    ↓
Compute metrics from df_filtered
    ↓
Build charts from df_filtered
    ↓
Display table from df_filtered
    ↓
Download from df_filtered
```

Every step uses `df_filtered`. Raw `df` is only used for baseline comparisons 
(e.g., delta in st.metric vs. full dataset average).

### Upwork Angle

A Streamlit app demo link = the strongest possible Upwork proposal attachment.
Instead of "I can do data analysis", you say:
*"Here's a live dashboard I built for a similar retail dataset: [link]"*
Conversion rate from proposals with live demos is 3–5x higher.

### Interview Answer Template

> "When a client needs to explore their data interactively, I build a Streamlit dashboard —
> it's a Python web app with live filters, KPI cards, and export functionality. 
> Unlike a static PDF, the client can slice by region, date range, or product category 
> themselves and download exactly what they need. I typically deploy it to Streamlit 
> Community Cloud so they get a shareable URL within the same day."


---
## Section 3 — Answer Key
*Attempt every task independently before opening this section.*

In [ ]:
# ANSWER KEY — Day 93
# ─────────────────────────────────────────────────────
# Full Day93_dashboard.py — read after attempting

ANSWER_KEY = """
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── App config (MUST be first Streamlit call) ─────────────────────────────────
st.set_page_config(
    page_title="TechMart India | Analytics Dashboard",
    page_icon="📊",
    layout="wide"
)

# ── Bonus B1: Custom CSS header ───────────────────────────────────────────────
st.markdown('''
<div style="background-color:#1F3864; padding:16px; border-radius:8px; text-align:center">
    <h2 style="color:white; margin:0">📊 TechMart India Analytics Dashboard</h2>
    <p style="color:#D6E4F0; margin:4px 0 0 0">Powered by Python + Streamlit</p>
</div>
''', unsafe_allow_html=True)

st.markdown("")  # spacer

st.subheader("Interactive sales analytics for TechMart India — filter, explore, export.")
st.divider()

# ── T2: Load data ─────────────────────────────────────────────────────────────
@st.cache_data
def load_data():
    df = pd.read_csv('techmart_day93.csv', parse_dates=['date'])
    df = df.dropna(subset=['order_id']).reset_index(drop=True)
    return df

df = load_data()

# ── T1/T2: Sidebar filters ────────────────────────────────────────────────────
st.sidebar.header("🎛️ Filters")

regions_available   = sorted(df['region'].unique())
categories_available = sorted(df['category'].unique())
min_date = df['date'].min().date()
max_date = df['date'].max().date()

selected_regions = st.sidebar.multiselect('Region', regions_available,
                                           default=regions_available)
selected_cats    = st.sidebar.multiselect('Category', categories_available,
                                           default=categories_available)
date_range       = st.sidebar.date_input('Date Range',
                                          value=[min_date, max_date],
                                          min_value=min_date, max_value=max_date)

# Guard: ensure date_range has two values
if len(date_range) == 2:
    start_date, end_date = date_range
else:
    start_date, end_date = min_date, max_date

# Apply filters
df_filtered = df[
    (df['region'].isin(selected_regions)) &
    (df['category'].isin(selected_cats)) &
    (df['date'].dt.date >= start_date) &
    (df['date'].dt.date <= end_date)
].copy()

st.sidebar.markdown(f'**{len(df_filtered)} orders** match filters')

# ── T3: KPI cards ────────────────────────────────────────────────────────────
col1, col2, col3 = st.columns(3)

total_revenue  = df_filtered['revenue'].sum()
total_orders   = len(df_filtered)
filtered_aov   = df_filtered['revenue'].mean() if total_orders > 0 else 0
baseline_aov   = df['revenue'].mean()
delta_aov      = filtered_aov - baseline_aov

col1.metric('💰 Total Revenue',    f'₹{total_revenue:,.0f}')
col2.metric('📦 Total Orders',     f'{total_orders:,}')
col3.metric('🧾 Avg Order Value',  f'₹{filtered_aov:,.0f}',
            delta=f'₹{delta_aov:+,.0f} vs baseline')

st.divider()

# ── T4: Bar chart — Revenue by Category ──────────────────────────────────────
left_col, right_col = st.columns(2)

with left_col:
    st.subheader('Revenue by Category')
    df_cat = df_filtered.groupby('category')['revenue'].sum().sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.barh(df_cat.index, df_cat.values, color='#1F3864')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))
    ax.set_title('Revenue by Category', fontsize=12, fontweight='bold', color='#1F3864')
    ax.set_xlabel('Revenue (₹)')
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    st.pyplot(fig)
    top_cat = df_cat.idxmax()
    top_rev = df_cat.max()
    st.markdown(f'**{top_cat}** leads with ₹{top_rev:,.0f} in revenue, driven by '
                f'high unit prices in this segment, suggesting prioritised stock '
                f'replenishment and targeted promotions for {top_cat}.')

# ── Bonus B2: Pie chart — Revenue by Segment ─────────────────────────────────
with right_col:
    st.subheader('Revenue by Customer Segment')
    df_seg = df_filtered.groupby('customer_segment')['revenue'].sum()
    fig3, ax3 = plt.subplots(figsize=(7, 4))
    colors = ['#1F3864', '#2E5596', '#4A90D9', '#A8C4E0']
    ax3.pie(df_seg.values, labels=df_seg.index, autopct='%1.1f%%',
            colors=colors, startangle=140,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
    ax3.set_title('Revenue by Customer Segment', fontsize=12,
                  fontweight='bold', color='#1F3864')
    plt.tight_layout()
    st.pyplot(fig3)
    top_seg = df_seg.idxmax()
    top_seg_rev = df_seg.max()
    st.markdown(f'**{top_seg}** customers contribute ₹{top_seg_rev:,.0f} '
                f'— run targeted retention campaigns for this segment.')

st.divider()

# ── T5: Line chart — Monthly Revenue Trend ───────────────────────────────────
st.subheader('Monthly Revenue Trend')
df_filtered2 = df_filtered.copy()
df_filtered2['month'] = df_filtered2['date'].dt.to_period('M')
df_monthly   = df_filtered2.groupby('month')['revenue'].sum().reset_index()
df_monthly['month'] = df_monthly['month'].astype(str)

fig2, ax2 = plt.subplots(figsize=(10, 4))
ax2.plot(df_monthly['month'], df_monthly['revenue'],
         marker='o', color='#2E5596', linewidth=2)
ax2.fill_between(df_monthly['month'], df_monthly['revenue'],
                 alpha=0.15, color='#2E5596')
ax2.set_title('Monthly Revenue Trend', fontsize=12, fontweight='bold', color='#1F3864')
ax2.set_xlabel('Month')
ax2.set_ylabel('Revenue (₹)')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'₹{y/1000:.0f}K'))
ax2.spines[['top', 'right']].set_visible(False)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
st.pyplot(fig2)

if not df_monthly.empty:
    peak_idx   = df_monthly['revenue'].idxmax()
    peak_month = df_monthly.loc[peak_idx, 'month']
    peak_rev   = df_monthly.loc[peak_idx, 'revenue']
    st.markdown(f'**{peak_month}** was the peak month at ₹{peak_rev:,.0f}, '
                f'suggesting demand concentration — plan inventory and campaigns '
                f'to sustain momentum in subsequent months.')

st.divider()

# ── T6: Data table ───────────────────────────────────────────────────────────
st.subheader('📋 Order-Level Detail')
st.caption(f'Showing {len(df_filtered):,} orders based on current filters')
display_cols = df_filtered[['order_id','date','region','category','product',
                             'quantity','unit_price','revenue','customer_segment']].copy()
display_cols.columns = ['Order ID','Date','Region','Category','Product',
                        'Qty','Unit Price (₹)','Revenue (₹)','Segment']
st.dataframe(display_cols, use_container_width=True, height=300)

# ── T7: Download button ───────────────────────────────────────────────────────
csv_bytes = df_filtered.to_csv(index=False).encode('utf-8')
st.download_button(
    label='⬇️ Download Filtered Data (CSV)',
    data=csv_bytes,
    file_name='techmart_filtered_report.csv',
    mime='text/csv',
    help='Downloads the current filtered view as a CSV file'
)

st.caption('TechMart India Analytics Dashboard | Built with Streamlit | deepanshu0110')
"""

print("Answer key loaded.")
print("Full dashboard = 80 pts + 10 bonus if both B1 (CSS header) and B2 (pie chart) are included.")


---
## Section 4 — Scoring Rubric

| Task | Criteria | Points |
|------|----------|--------|
| **T1** | `set_page_config` first (2), title with emoji (2), subheader is a sentence not a label (2), `st.divider()` (1), sidebar header (1) | 8 |
| **T2** | `dropna(subset=['order_id'])` present (3), all 3 filters built correctly (3), `df_filtered` correctly constructed with all 3 conditions (4), sidebar row count shown (2) | 12 |
| **T3** | `st.columns(3)` layout (2), total revenue correct ₹ format (3), total orders correct (3), avg order value correct with delta (4) | 12 |
| **T4** | `groupby` correct (2), horizontal bar chart rendered (3), ₹K x-axis formatter (2), NRA insight with exact number from `df_cat` (5) | 12 |
| **T5** | `to_period('M')` used (2), line + fill plotted (3), ₹K y-axis formatter (2), NRA insight uses `idxmax()` value not a guess (5) | 12 |
| **T6** | Columns renamed for client readability (3), `use_container_width=True` (2), `height=300` (1), caption with row count (2) | 8 |
| **T7** | `to_csv().encode('utf-8')` (3), `download_button` with correct `mime` (3), footer caption (2) | 8 |
| **W1a** | Mentions Streamlit + why better than PDF + one specific feature + business value (4) | 4 |
| **W1b** | Sounds like a portfolio README, not homework. Includes: project name, what it does, stack, how to run (4) | 4 |
| **B★** | CSS header banner with correct colours (5) + Pie chart with NRA insight (5) | +10 |

### Common Deductions
| Error | Deduction |
|-------|-----------|
| Missing `dropna` before any analysis | −3 |
| Charts use raw `df` instead of `df_filtered` | −4 per chart |
| NRA insight cites wrong number (not from printed output) | −3 per insight |
| `set_page_config` not the first Streamlit call | −2 |
| W1 answer sounds like a student, not an analyst | −2 |

---
## Interview Framing — Day 93

> *"An interviewer asks: 'How do you present your analysis to non-technical clients?'"*

**Strong answer:**
> "I build interactive Streamlit dashboards. The client gets a web app with sidebar filters —
> region, category, date range — and KPI cards at the top showing revenue, orders, and 
> average order value. Charts update live as they change filters, and there's a download 
> button to export exactly what they're looking at. I deploy it to Streamlit Community Cloud
> so they get a URL, not a file attachment. It takes a client from 'I need to understand 
> my data' to 'I can explore my data myself' — which is worth significantly more than 
> a static PDF."


---
## Week 3 Scorecard (Month 5)

| Day | Topic | Score |
|-----|-------|-------|
| 90 | SQL → Python Pipeline | 110/100 ✅ |
| 91 | Python → BI Integration | Pending |
| 92 | End-to-End Client Report Pipeline | 72/80 + 5★ |
| **93** | **Streamlit Interactive Dashboard** | **Pending** |

**Pattern to fix this week:** Print the table → read the exact value → then write the insight.
The South/East error from Day 92 costs you 5 bonus points and is 100% preventable by one `print()` call.
